# Notebook : EDA_verbatims.ipynb

## Authors : Groupe VA-AI — Engineering project 2026

### Participants : Joris LARMAILLARD-NOIREN

## Summary

1. Contexte
2. Chargement
3. Quality controls
4. Text preparation (corpus + NMF)
5. Verbatim statistics (lengths)
6. Non-informative + cleaning + re-fit topics
7. Construction of analysis dataset + export

---

## 1. Context and purpose of the EDA

In this notebook, we will analyze raw text data provided by the “Quality & KPI” department. The goal is to perform an exploratory analysis of the data to better understand it and adjust our strategy for implementing our AI models.

---

## 2. Loading the data

In [1]:
### Modules importation
import re
from typing import Any
from collections import Counter

import pandas as pd
import numpy as np
from pathlib import Path

from pandas import DataFrame
from unidecode import unidecode
from sklearn.decomposition import NMF
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
### File path
xlsx_path = Path("../../data/raw/BDD PROJET VA AI 070126.xlsx")

In [3]:
### Loading the data in a dataframe
df = pd.read_excel(xlsx_path, sheet_name="BDD", engine="openpyxl", dtype=str)

### Copying the dataset
df_full = df.copy()

In [4]:
### Preview
df.head()

,Année d'enquete,Ancienneté,Natio,Cursus,StudentType,Cycle,Genre,AcademicLevel,Campus,A répondu,Date de réponse,"Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Dans l’ensemble, je suis satisfait de l’Efrei","Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Je suis satisfait de ma formation à l’Efrei",Q58_Quelles seraient vos suggestions d’amélioration pour l’Efrei ?,Q59_Quels autres messages souhaitez-vous adresser à l’Efrei ?
0,24-25,New,France,PGE,FI,Cycle L,H,I1,VILLEJUIF,Oui,2025-02-17 05:20:00,Plutôt d'accord,Plutôt d'accord,D/A,D/A
1,24-25,Ancien,France,Pex,FA,Pex,H,M1PEX,VILLEJUIF,Oui,2025-02-17 05:18:00,Plutôt d'accord,Plutôt d'accord,D/A,D/A
2,24-25,Ancien,France,PGE,FA,Cycle M,H,I2,VILLEJUIF,Oui,2025-02-17 05:24:00,Tout à fait d'accord,Tout à fait d'accord,D/A,D/A
3,24-25,Ancien,France,Pex,FA,Pex,F,M1PEX,VILLEJUIF,Oui,2025-02-17 05:34:00,Plutôt d'accord,Tout à fait d'accord,D/A,D/A
4,24-25,New,France,PGE,FA,Cycle L,H,I1,VILLEJUIF,Oui,2025-02-17 05:25:00,Plutôt d'accord,Plutôt d'accord,Les notes d’examens doivent être communiquées ...,D/A


In [5]:
### Retrieving questions columns
questions_cols = [c for c in df.columns if c.startswith("Q")]

df = df_full[questions_cols].copy()

In [6]:
### Preview
df.head()

,"Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Dans l’ensemble, je suis satisfait de l’Efrei","Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Je suis satisfait de ma formation à l’Efrei",Q58_Quelles seraient vos suggestions d’amélioration pour l’Efrei ?,Q59_Quels autres messages souhaitez-vous adresser à l’Efrei ?
0,Plutôt d'accord,Plutôt d'accord,D/A,D/A
1,Plutôt d'accord,Plutôt d'accord,D/A,D/A
2,Tout à fait d'accord,Tout à fait d'accord,D/A,D/A
3,Plutôt d'accord,Tout à fait d'accord,D/A,D/A
4,Plutôt d'accord,Plutôt d'accord,Les notes d’examens doivent être communiquées ...,D/A


---

**Function utils to read a `.xls` file**

In [7]:
### Function : read_file
def read_excel_auto(path, sheet_name):
    """
    Function to read automatically a `.xlsx` file and select the sheet that contains the questions.
    :param path: File path
    :param sheet_name: Name of sheet that contains the questions
    :return:
    """
    ### File path
    path = Path(path)

    ### Case : it's a xlsx file
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl", dtype=str)
    ### Case : it's a xls file
    elif path.suffix.lower() == ".xls":
        return pd.read_excel(path, sheet_name=sheet_name, engine="xlrd", dtype=str)
    ### Case : it's a binary file
    elif path.suffix.lower() == ".xlsb":
        return pd.read_excel(path, sheet_name=sheet_name, engine="pyxlsb", dtype=str)
    else:
        raise ValueError(f"Extension not supported : {path.suffix}")

---

## 3. Quality controls

This section defines some helpers to normalize the data.

**Function helpers**

In [8]:
### Function : normalize_text
def normalize_text(s: str) -> str:
    """
    Function to normalize the text in a correct string format.
    :param s: String to normalize
    :return: Normalized string
    """
    if not isinstance(s, str):
        return ""
    ### Removing accent, lowering the case and text trimming
    text = unidecode(s.lower()).strip()

    ### Removing multiple withe spaces
    text = re.sub(r"\s+", " ", text)

    ### Removing punctuation
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)

    ### Normalizing line break
    text = text.replace(r"_x000d_", " ")
    text = text.replace(r"x000d", " ")

    return text

In [9]:
### Function : normalize_col_name
def normalize_col_name(df_old: pd.DataFrame) -> tuple[DataFrame, dict[Any, Any]]:
    """
    Function to rename columns in an understable string format of a given dataframe.
    :param df_old: Original dataframe
    :return: Normalized name
    """
    ### Columns dictionary
    mapping = {}

    ### Counter to avoid identical name
    counts = Counter()

    ### Retrieving the question number from the original column name
    for col in df_old.columns:
        ### Saved format : "Q{qst_number}_"
        m = re.match(r"^(Q\d+)_", str(col))
        if not m:
            continue

        base = m.group(1)
        counts[base] += 1
        new_name = base if counts[base] == 1 else f"{base}_{counts[base]}"
        mapping[col] = new_name

    ### Apply modification
    df_new = df_old.rename(columns=mapping)
    return df_new, mapping

In [10]:
### Function : normalize_da
def normalize_da(x: str) -> str:
    """
    Function to normalize the "d/a" text in a string format of a given dataframe.
    :param x: The string to normalize
    :return: The normalized string
    """
    if x is None:
        return ""
    s = str(x).strip()

    ### String normalization by transforming "D/A"
    s_norm = re.sub(r"\s+", "", s.lower())
    if s_norm in {"d/a", "da"}:
        return ""
    return s

In [11]:
### Normalize the data
mapping = dict()
df, mapping = normalize_col_name(df)

for c in df.columns:
    df[c] = df[c].apply(normalize_text)

In [12]:
### Old name mapping with the renamed column
mapping

{'Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Dans l’ensemble, je suis satisfait de l’Efrei': 'Q1',
 'Q1_Pour chaque item proposé, merci de cocher la case correspondant à votre niveau d’accord. (Pas du tout d’accord à tout à fait d’accord)._Je suis satisfait de ma formation à l’Efrei ': 'Q1_2',
 'Q58_Quelles seraient vos suggestions d’amélioration pour l’Efrei ? ': 'Q58',
 'Q59_Quels autres messages souhaitez-vous adresser à l’Efrei ?': 'Q59'}

In [13]:
### Preview
df.head()

,Q1,Q1_2,Q58,Q59
0,plutot d accord,plutot d accord,d a,d a
1,plutot d accord,plutot d accord,d a,d a
2,tout a fait d accord,tout a fait d accord,d a,d a
3,plutot d accord,tout a fait d accord,d a,d a
4,plutot d accord,plutot d accord,les notes d examens doivent etre communiquees ...,d a


---

## 4. Text preparation

**Building a corpus based on open-ended questions**

In [14]:
### Retrieving the open-ended questions
open_cols = [c for c in df.columns if c.startswith("Q58") or c.startswith("Q59")]

### Building the corpus
corpus = (
    df[open_cols]
    .map(normalize_da)
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [15]:
### Preview
corpus.head()

0                                                     
1                                                     
2                                                     
3                                                     
4    les notes d examens doivent etre communiquees ...
dtype: object

In [16]:
### Removing empty lines from the corpus
mask_nonempty = corpus != ""
### Here we don't reset the index
corpus_clean = corpus[mask_nonempty]
idx_clean = corpus_clean.index

**Corpus vectorization**

In [17]:
### Defining the vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=stopwords.words("french"),
    max_features=5000,
    ngram_range=(1,2),
    min_df=3
)

### Fitting the data
X = vectorizer.fit_transform(corpus_clean)

**Topics analysis**

In [18]:
### Retrieving the topics via NMF
n_topics = 20

### Defining our topics modeler
nmf = NMF(n_components=n_topics, random_state=42)
W = nmf.fit_transform(X)
H = nmf.components_

terms = vectorizer.get_feature_names_out()

**Top words by topic**

In [19]:
### Top words by topic
top_words_by_topic = {}
for k in range(n_topics):
    top_idx = H[k].argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_idx]
    top_words_by_topic[k] = top_terms
    print(f"Topic {k}: {', '.join(top_terms)}")

Topic 0: bien, faire, meme, tres, tout, fait, beaucoup, efrei, ca, trop
Topic 1: ras, ras ras, qualite enseignement, plus pratique, enseignement, formations, plus vie, pratique, qualite, plus aide
Topic 2: xp, learning, learning xp, points, programme learning, programme, points learning, revoir, obligatoire, systeme
Topic 3: rien, rien rien, rien dire, dire, rien particulier, particulier, signaler, rien ajouter, sws, ecoutez
Topic 4: ecoute, plus ecoute, ecoute etudiants, etre, etre plus, etudiants, ecoute eleves, plus, etre ecoute, meilleure ecoute
Topic 5: plus, moins, pratique, plus pratique, plus cours, peu plus, bordeaux, peu, faire plus, plus evenements
Topic 6: communication, meilleure communication, communication entre, entre, meilleure, ameliorer communication, ameliorer, plus communication, moins, cours communication
Topic 7: merci, merci tout, tout, aucunes, suggestions, viennent, cette, distributeur, merci beaucoup, meilleurs
Topic 8: aucun, aucune, aucune aucun, aucun aucu

**Most recurring topics**

In [20]:
### Most recurring topics
dominant_topic = W.argmax(axis=1)
counts = np.bincount(dominant_topic, minlength=n_topics)
top_order = counts.argsort()[::-1]

print("\n--- Most recurring topics ---")
for k in top_order:
    print(f"Topic {k}: {counts[k]} docs | label auto : {' / '.join(top_words_by_topic[k][:3])}")


--- Most recurring topics ---
Topic 0: 382 docs | label auto : bien / faire / meme
Topic 9: 285 docs | label auto : cours / plus cours / visio
Topic 17: 263 docs | label auto : salles / campus / wifi
Topic 5: 254 docs | label auto : plus / moins / pratique
Topic 16: 253 docs | label auto : etudiants / alternance / stage
Topic 11: 213 docs | label auto : niveau / intervenants / professeurs
Topic 18: 212 docs | label auto : ecole / bonne / bonne ecole
Topic 19: 192 docs | label auto : avoir / notes / avoir plus
Topic 12: 156 docs | label auto : eleves / ecouter / ecouter eleves
Topic 2: 139 docs | label auto : xp / learning / learning xp
Topic 15: 136 docs | label auto : temps / emploi / emploi temps
Topic 6: 108 docs | label auto : communication / meilleure communication / communication entre
Topic 14: 101 docs | label auto : vie / associative / vie associative
Topic 3: 95 docs | label auto : rien / rien rien / rien dire
Topic 7: 70 docs | label auto : merci / merci tout / tout
Topic 4

In [21]:
### Examples of dominant verbatim statements
print("\n--- Examples of dominant verbatim statements (top verbatims) ---")
for k in top_order[:10]:
    print(f"\n=== Topic {k} ({counts[k]} docs) ===")
    print("Key words :", ", ".join(top_words_by_topic[k][:10]))

    ### Retrieving the top 3 docs for this topic
    top_docs = np.argsort(W[:, k])[-3:][::-1]
    for i in top_docs:
        txt = corpus_clean.iloc[i]
        ### Display an excerpt to avoid dumping all the text
        print("-", txt[:250], "..." if len(txt) > 250 else "")


--- Examples of dominant verbatim statements (top verbatims) ---

=== Topic 0 (382 docs) ===
Key words : bien, faire, meme, tres, tout, fait, beaucoup, efrei, ca, trop
- lorsque j ai envoye une demande au service de comptabilite celui ci m a repondu 2 mois plus tard le wifi du nouveau batiment marche tres peu souvent ce qui est embetant pour les cours d informatique meme si les locaux sont tres bien je trouve egaleme ...
- prendre plus en compte l avis des etudiants on fait plein d enquetes de satisfaction et on a l impression que rien de change aussi sur les nouveaute mises en places qui ont ete mal recues le nouveau systeme d emargement la nouvelle plateforme d evalu ...
- je n ai pas d observations particulieres a faire si ce n est que les cours de formation generale abordent souvent les memes thematiques d une annee a l autre ou alors le travail demande peut largement se faire en autonomie sous la forme d un tpa je p ...

=== Topic 9 (285 docs) ===
Key words : cours, plus cours, v

---

## 5. Verbatim statistics

**Mean length of verbatims**

In this section we'll analyse the structure of the verbatims.

In [22]:
### Character/word length
char_len = corpus_clean.str.len()
word_len = corpus_clean.str.split().str.len()

In [23]:
print("Nb docs:", len(corpus_clean))
print("Average length (characters) :", round(char_len.mean(), 1))
print("Median length (characters) :", round(char_len.median(), 1))
print("Average length (words) :", round(word_len.mean(), 1))
print("Median length (words) :", round(word_len.median(), 1))

Nb docs: 3125
Average length (characters) : 344.2
Median length (characters) : 170.0
Average length (words) : 61.0
Median length (words) : 29.0


Mean > median (`61` vs. `29` words) = highly skewed distribution : many short responses and a few very long ones.

- **Median length (characters) : `170` characters**
- **Median length (words) : `29` words**

In [24]:
### Top longest verbatims (wordwise)
print("\nTop 10 longest words :")
print(word_len.sort_values(ascending=False).head(10))


Top 10 longest words :
2649     1132
685       882
3812      838
7997      784
994       755
10675     706
12345     704
12371     702
2018      663
4553      602
dtype: int64


---

## 6. Non-informative + cleanup + re-fit topics

**Analysis of usable verbatim statements**

**Case of non-informative verbatims**

In [25]:
### Regex to detect a non-informative verbatim
NON_INFO_PATTERNS = re.compile(r"^\s*(ras|rien|rien a dire|neant|n/a|na|non|ok|rien particulier|rien ajouter)\s*$", re.IGNORECASE)

In [26]:
is_non_info = corpus_clean.apply(lambda s: bool(NON_INFO_PATTERNS.match(s)))
print("Non-informatives (%):", round(is_non_info.mean()*100, 2))

Non-informatives (%): 0.13


**Removing the non-informative statements**

In [27]:
### Redefining the vectorizer
base_sw = set(stopwords.words("french"))
custom_sw = base_sw | {
    "ras", "rien", "aucun", "aucune", "neant", "n/a", "na", "merci",
    "ok", "d", "a", "da", "x000d", "rien particulier", "rien ajouter"
}

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=list(custom_sw),
    max_features=5000,
    ngram_range=(1,2),
    min_df=3
)

In [28]:
### Recreating the corpus with all the meaningful statements
corpus_thematic = corpus_clean[~is_non_info]
idx_thematic = corpus_thematic.index

In [29]:
### Fitting the data
X = vectorizer.fit_transform(corpus_thematic)

W = nmf.fit_transform(X)
H = nmf.components_

terms = vectorizer.get_feature_names_out()

### Top words by topic
top_words_by_topic = {}
for k in range(n_topics):
    top_idx = H[k].argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_idx]
    top_words_by_topic[k] = top_terms
    print(f"Topic {k}: {', '.join(top_terms)}")

### Most recurring topics
dominant_topic = W.argmax(axis=1)
topic_series = pd.Series(dominant_topic, index=idx_thematic, name="topic")
counts = np.bincount(dominant_topic, minlength=n_topics)
top_order = counts.argsort()[::-1]

print("\n--- Most recurring topics ---")
for k in top_order:
    print(f"Topic {k}: {counts[k]} docs | label auto : {' / '.join(top_words_by_topic[k][:3])}")

/opt/miniconda3/envs/va-ai-venv/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ajouter', 'particulier'] not in stop_words.
  warnings.warn(


Topic 0: tout, faire, bien, efrei, tres, meme, fait, ca, annee, car
Topic 1: xp, learning, learning xp, points, programme learning, programme, points learning, revoir, obligatoire, systeme
Topic 2: plus, moins, pratique, plus pratique, plus cours, peu plus, bordeaux, peu, faire plus, plus evenements
Topic 3: cours, plus cours, visio, moins, moins cours, cours visio, mettre, trop, cours anglais, heures
Topic 4: communication, meilleure communication, meilleure, communication entre, entre, plus communication, moins, ameliorer communication, cours communication, communication interne
Topic 5: sais, sais sais, message, tres bien, bien, signer, bonne continuation, continuation, contacter, tres
Topic 6: etudiants, ecoute, ecoute etudiants, plus ecoute, etre, etre plus, ecoute eleves, meilleure ecoute, etre ecoute, leurs
Topic 7: organisation, meilleure, meilleure organisation, meilleure communication, gestion, avoir meilleure, organisation niveau, meilleur organisation, informations, examens

**Construction of the analysis dataset**

In [30]:
### Answer mapping
sat_map = {
  "pas du tout d accord": 1,
  "plutot pas d accord": 2,
  "plutot d accord": 3,
  "tout a fait d accord": 4,
}

In [31]:
### Full dataset with text features and topics
analysis = df_full.copy()

In [32]:
### Text with original index
analysis.loc[idx_clean, "corpus_clean"] = corpus_clean
analysis.loc[idx_clean, "n_chars"] = corpus_clean.str.len()
analysis.loc[idx_clean, "n_words"] = corpus_clean.str.split().str.len()
analysis.loc[idx_clean, "is_non_info"] = is_non_info

In [33]:
### Topic only for idx_thematic
analysis.loc[idx_thematic, "topic"] = topic_series

In [34]:
### Overall satisfaction
analysis["sat_global"] = df["Q1"].map(sat_map)

In [35]:
### Subtable : only rows where there is a topic
analysis_topics = analysis.loc[idx_thematic].copy()

---

## 7. Construction of analysis dataset + export

**Final exportation**

In [36]:
out = Path("../../data/processed/analysis_topics.parquet")
out.parent.mkdir(parents=True, exist_ok=True)

analysis_topics.to_parquet(out, index=True)
print("Saved:", out)

Saved: ../../data/processed/analysis_topics.parquet
